# Fine-tuning ESM-C with LoRA for Druggability PredictionAdapts the ESM-C LoRA fine-tuning template for binary druggability classification on 3,854 human proteins. Built on top of Dr. Henry Kilgore's `esmc_finetune.ipynb` template; main differences are binary classification (instead of multi-class enzyme prediction), AUC evaluation (instead of accuracy only), and the custom druggability dataset.For background on LoRA, see the [PEFT documentation](https://huggingface.co/docs/peft/package_reference/lora).

## Imports

In [ ]:
# !pip install peft transformers accelerate

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from peft import LoraConfig, get_peft_model
from sklearn.metrics import roc_auc_score, confusion_matrix
from tqdm.auto import tqdm
from transformers import AutoTokenizer, ESMCForSequenceClassification

## Load the druggability datasetThe dataset is `data/druggability_dataset.csv`, 3,854 human proteins with binary `druggable` labels (1,123 positive, 2,731 negative). See `data/README.md` for full provenance.

In [ ]:
DATASET_PATH = Path("./data/druggability_dataset.csv")

df = pd.read_csv(DATASET_PATH)
print(f"Total proteins: {len(df):,}")
print(f"Positive (druggable): {int(df['druggable'].sum()):,}")
print(f"Negative: {int((1 - df['druggable']).sum()):,}")
print(f"Positive rate: {df['druggable'].mean():.1%}")
df.head()

## Stratified train / dev / test split70/15/15 split, stratified by the binary label so all three subsets preserve the ~29% positive class ratio. Fixed seed for reproducibility.

In [ ]:
SPLIT_SEED = 0TRAIN_FRACTION = 0.70DEV_FRACTION = 0.15# Test fraction is the remainder (0.15)rng = np.random.default_rng(SPLIT_SEED)def stratified_split(df, train_frac, dev_frac, rng):    train_idx, dev_idx, test_idx = [], [], []    for label_value in sorted(df['druggable'].unique()):        class_positions = df.index[df['druggable'] == label_value].to_numpy()        rng.shuffle(class_positions)        n = len(class_positions)        n_train = int(round(n * train_frac))        n_dev = int(round(n * dev_frac))        train_idx.extend(class_positions[:n_train])        dev_idx.extend(class_positions[n_train:n_train + n_dev])        test_idx.extend(class_positions[n_train + n_dev:])    return sorted(train_idx), sorted(dev_idx), sorted(test_idx)train_idx, dev_idx, test_idx = stratified_split(df, TRAIN_FRACTION, DEV_FRACTION, rng)train_df = df.loc[train_idx].reset_index(drop=True)dev_df = df.loc[dev_idx].reset_index(drop=True)test_df = df.loc[test_idx].reset_index(drop=True)print(f"Train: {len(train_df):,}  (positive rate: {train_df['druggable'].mean():.1%})")print(f"Dev:   {len(dev_df):,}  (positive rate: {dev_df['druggable'].mean():.1%})")print(f"Test:  {len(test_df):,}  (positive rate: {test_df['druggable'].mean():.1%})")

In [ ]:
train_sequences = train_df["sequence"].tolist()train_labels = train_df["druggable"].to_numpy(dtype=np.float32)dev_sequences = dev_df["sequence"].tolist()dev_labels = dev_df["druggable"].to_numpy(dtype=np.float32)test_sequences = test_df["sequence"].tolist()test_labels = test_df["druggable"].to_numpy(dtype=np.float32)

## Visualize sequence length distributionESM-C's context length is 1024 tokens. Sequences longer than this get truncated; worth knowing how many that affects.

In [ ]:
MAX_SEQ_LEN = 1024fig, ax = plt.subplots(figsize=(10, 4))ax.hist(df['length'], bins=80, color='steelblue', alpha=0.8)ax.axvline(MAX_SEQ_LEN, color='crimson', linestyle='--', label=f'truncation = {MAX_SEQ_LEN}')ax.set_xlabel('Protein length (residues)')ax.set_ylabel('Number of proteins')ax.set_title('Sequence length distribution')ax.legend()plt.tight_layout()plt.show()n_truncated = int((df['length'] > MAX_SEQ_LEN).sum())print(f"Proteins exceeding {MAX_SEQ_LEN} residues: {n_truncated:,} ({n_truncated / len(df):.1%})")

## Load tokenizer and model with LoRALoads the pretrained ESM-C backbone with a fresh classification head (`num_labels=1` for binary), then wraps with LoRA adapters. Only LoRA parameters plus the new classifier head are trained; backbone weights stay frozen.The classification head uses a single logit + `BCEWithLogitsLoss` rather than 2-way softmax. This is the standard pattern for binary classification in HuggingFace and produces a continuous probability via `sigmoid(logit)`.

In [ ]:
MODEL_PATH = "Biohub/ESMC-300M"tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)model = ESMCForSequenceClassification.from_pretrained(    MODEL_PATH,    num_labels=1,                     # binary classification with a single logit    problem_type="single_label_classification",    device_map="auto",)

In [ ]:
LORA_RANK = 8LORA_ALPHA = 16# Matches the targets used in the original esmc_finetune template:# - target_modules covers Linear submodules# - target_parameters covers bare nn.Parameter tensors (fused QKV / FFN weights)lora_config = LoraConfig(    r=LORA_RANK,    lora_alpha=LORA_ALPHA,    lora_dropout=0.01,    target_modules=["out_proj"],    target_parameters=["layernorm_qkv.weight", "ffn.fc1_weight", "ffn.fc2_weight"],    modules_to_save=["classifier"],)model = get_peft_model(model, lora_config)print("\n=== Trainable Parameters ===")model.print_trainable_parameters()device = next(model.parameters()).deviceprint(f"\nDevice: {device}")

## Helpers for training and evaluation

In [ ]:
EVAL_BATCH_SIZE = 16def make_batch(sequences, labels):    inputs = tokenizer(        sequences,        return_tensors="pt",        padding=True,        truncation=True,        max_length=MAX_SEQ_LEN,    )    return {        "input_ids": inputs["input_ids"].to(device),        "attention_mask": inputs["attention_mask"].to(device),        "labels": torch.tensor(labels, dtype=torch.float32, device=device).unsqueeze(-1),    }@torch.no_grad()def evaluate(model, name, sequences, labels, batch_size=EVAL_BATCH_SIZE):    """Returns predicted probabilities, loss, accuracy, AUC."""    all_probs = np.empty(len(sequences), dtype=np.float32)    all_losses = []    model.eval()    with torch.autocast(device_type="cuda", dtype=torch.bfloat16):        for start in range(0, len(sequences), batch_size):            end = start + batch_size            batch = make_batch(sequences[start:end], labels[start:end])            outputs = model(**batch)            all_losses.append(outputs.loss.item() * (end - start))            probs = torch.sigmoid(outputs.logits).squeeze(-1).float().cpu().numpy()            all_probs[start:end] = probs    loss = float(np.sum(all_losses) / len(sequences))    preds = (all_probs >= 0.5).astype(np.int64)    acc = float((preds == labels.astype(np.int64)).mean())    auc = float(roc_auc_score(labels, all_probs))    print(f"{name}: loss={loss:.4f}  acc={acc:.4f}  auc={auc:.4f}")    return all_probs, loss, acc, aucdef plot_training_metrics(losses, accuracies, val_history, window=20):    fig, axes = plt.subplots(1, 3, figsize=(15, 4))    # Smoothed training loss    smoothed_loss = pd.Series(losses).rolling(window, min_periods=1).mean()    axes[0].plot(losses, alpha=0.3, label='step')    axes[0].plot(smoothed_loss, color='crimson', label=f'rolling mean ({window})')    axes[0].set_xlabel('Step')    axes[0].set_ylabel('Train loss')    axes[0].set_title('Training loss')    axes[0].legend()    # Smoothed training accuracy    smoothed_acc = pd.Series(accuracies).rolling(window, min_periods=1).mean()    axes[1].plot(accuracies, alpha=0.3, label='step')    axes[1].plot(smoothed_acc, color='seagreen', label=f'rolling mean ({window})')    axes[1].set_xlabel('Step')    axes[1].set_ylabel('Train accuracy')    axes[1].set_title('Training accuracy')    axes[1].legend()    # Validation AUC over checkpoints    if val_history:        steps = [v['step'] for v in val_history]        aucs = [v['auc'] for v in val_history]        axes[2].plot(steps, aucs, marker='o', color='navy')        axes[2].set_xlabel('Step')        axes[2].set_ylabel('Dev AUC')        axes[2].set_title('Dev AUC during training')        axes[2].axhline(0.815, color='gray', linestyle='--', alpha=0.5, label='Phase 1 baseline (0.815)')        axes[2].legend()    plt.tight_layout()    plt.show()

## Training loop

In [ ]:
# Tune NUM_TRAINING_STEPS based on convergence behavior.# ~5000 steps recommended for a meaningful finetune; 1000 used here for a quick smoke test.# 1000 steps takes ~4 minutes on an H100 per the original ESM-C template.NUM_TRAINING_STEPS = 1000BATCH_SIZE = 8LEARNING_RATE = 1e-4SEED = 0VAL_EVERY_STEPS = 250CHECKPOINT_DIR = Path("./checkpoints/best")CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
trainable = [p for p in model.parameters() if p.requires_grad]optimizer = torch.optim.AdamW(trainable, lr=LEARNING_RATE)rng = np.random.default_rng(SEED)n_train = len(train_sequences)losses = []accuracies = []val_history = []best_val_auc = -1.0postfix = {"train_loss": "n/a", "train_acc": "n/a", "val_auc": "n/a"}model.train()perm = rng.permutation(n_train)pbar = tqdm(range(NUM_TRAINING_STEPS), desc="training")for step in pbar:    start = (step * BATCH_SIZE) % n_train    if start + BATCH_SIZE > n_train:        perm = rng.permutation(n_train)        start = 0    idx = perm[start:start + BATCH_SIZE]    batch = make_batch(        [train_sequences[int(i)] for i in idx],        train_labels[idx],    )    with torch.autocast(device_type="cuda", dtype=torch.bfloat16):        outputs = model(**batch)        loss = outputs.loss    loss.backward()    optimizer.step()    optimizer.zero_grad()    probs = torch.sigmoid(outputs.logits).squeeze(-1).float()    preds = (probs >= 0.5).long()    targets = batch["labels"].squeeze(-1).long()    acc = (preds == targets).float().mean().item()    losses.append(loss.item())    accuracies.append(acc)    if (step + 1) % 20 == 0:        postfix["train_loss"] = f"{np.mean(losses[-20:]):.3f}"        postfix["train_acc"] = f"{np.mean(accuracies[-20:]):.2f}"        pbar.set_postfix(postfix)    if (step + 1) % VAL_EVERY_STEPS == 0 or step == NUM_TRAINING_STEPS - 1:        _, val_loss, val_acc, val_auc = evaluate(            model, f"dev @ step {step + 1}", dev_sequences, dev_labels        )        val_history.append({"step": step + 1, "loss": val_loss, "acc": val_acc, "auc": val_auc})        postfix["val_auc"] = f"{val_auc:.3f}"        pbar.set_postfix(postfix)        if val_auc > best_val_auc:            best_val_auc = val_auc            model.save_pretrained(str(CHECKPOINT_DIR))            print(f"  -> new best dev AUC {val_auc:.4f}; saved to {CHECKPOINT_DIR}")        model.train()print(f"\nBest dev AUC during training: {best_val_auc:.4f}")

In [ ]:
plot_training_metrics(losses, accuracies, val_history)

## Evaluation on the test setLoads the best checkpoint (by dev AUC) and evaluates on the held-out test set. Reports loss, accuracy, AUC, and the confusion matrix.

In [ ]:
# Reload the best checkpointfrom peft import PeftModelbase_model = ESMCForSequenceClassification.from_pretrained(    MODEL_PATH,    num_labels=1,    problem_type="single_label_classification",    device_map="auto",)best_model = PeftModel.from_pretrained(base_model, str(CHECKPOINT_DIR))best_model.eval()# Swap in the reloaded best model for evaluationtest_probs, test_loss, test_acc, test_auc = evaluate(    best_model, "test", test_sequences, test_labels)

In [ ]:
# Confusion matrix at threshold 0.5test_preds = (test_probs >= 0.5).astype(np.int64)cm = confusion_matrix(test_labels.astype(np.int64), test_preds)fig, ax = plt.subplots(figsize=(5, 4))im = ax.imshow(cm, cmap='Blues')ax.set_xticks([0, 1])ax.set_yticks([0, 1])ax.set_xticklabels(['not druggable', 'druggable'])ax.set_yticklabels(['not druggable', 'druggable'])ax.set_xlabel('Predicted')ax.set_ylabel('True')ax.set_title(f'Test confusion matrix\nAUC = {test_auc:.3f}  Acc = {test_acc:.3f}')for i in range(2):    for j in range(2):        ax.text(j, i, str(cm[i, j]), ha='center', va='center',                color='white' if cm[i, j] > cm.max() / 2 else 'black')plt.colorbar(im, ax=ax)plt.tight_layout()plt.show()

## Compare to Phase 1 baselineFor context, the frozen-embedding linear probe on ESM-2 layer 6 from Phase 1 got an AUC of 0.815 on the same dataset under random 5-fold cross-validation. The fine-tuned ESM-C number above should be compared against that with two caveats:1. **Different evaluation protocol.** The 0.815 was 5-fold CV across the full dataset; the number here is on a single held-out test split. Direct comparison is rough.2. **Family leakage applies to both.** Neither uses sequence-similarity-based splits yet. Real performance on novel families is almost certainly lower than either number suggests.The next planned step is MMseqs2-clustered splits, which should produce more honest performance estimates for both the linear probe and the fine-tuned model.

## Next steps- Run with `NUM_TRAINING_STEPS = 5000` once smoke test passes- Expand the dataset using Open Targets tractability annotations + tiered ChEMBL thresholds- Apply MMseqs2 sequence-similarity clustering for honest train/test splits- Sweep LoRA rank, alpha, and learning rate for the binary task- Deploy the trained model on the full 83k human proteome for the merged group database